<a href="https://colab.research.google.com/github/KarunyaVarma/Karunya-Varma/blob/main/DNN_GTPP_Regression1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -------------------------------------------------
# 1. Input Data & Reproducibility
# -------------------------------------------------

tf.keras.utils.set_random_seed(42)
data = pd.read_csv('/content/GTPP-DATASET.csv')

X = data[['AT', 'AP', 'DF', 'CH', 'GP', 'CPR', 'CPD', 'TTXM']].values
y = data['EP'].values

# -------------------------------------------------
# 2. Exact Paper Split (80:20) & MinMax Scaling
# -------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------
# 3. Final Comparison: Swish vs Tanh with LR Decay
# -------------------------------------------------

activations = ['swish','tanh','relu']
final_results = []

for act in activations:
    print(f"Running Final Experiment with activation='{act}' ...")
    tf.keras.utils.set_random_seed(42)

    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(8,)),
        tf.keras.layers.Dense(224, activation=act),
        tf.keras.layers.Dense(48, activation=act),
        tf.keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001603),
        loss='mse',
        metrics=['mae']
    )

    lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='loss',
        factor=0.5,
        patience=12,
        min_lr=1e-6,
        verbose=0
    )

    model.fit(
        X_train_scaled,
        y_train,
        epochs=200,
        batch_size=16,
        callbacks=[lr_scheduler],
        verbose=0
    )

    y_pred = model.predict(X_test_scaled, verbose=0).flatten()

    final_results.append({
        "Activation": act,
        "Params": model.count_params(),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2 Score": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(final_results)
print("\n==========================================================================")
print("               FINAL PHASE 3 COMPARISON (FULL 80% TRAIN)                  ")
print("==========================================================================")
print(results_df.to_string(index=False))
print("==========================================================================")